# K-means model selection for Beijing PM data

This notebook evaluates K-means on the Beijing PM observations using precipitation, humidity, temperature, dew point, instant wind speed, and pressure.

Instead of fixing $k=3$, it compares several values of $k$ with the elbow method and silhouette score.

In [2]:
import pandas as pd

beijing_df = pd.read_csv("BeijingPM20100101_20151231.csv")
# Keep only observations at hour 12
beijing_df = beijing_df[beijing_df["hour"] == 12]

feature_df = (
    beijing_df[["precipitation", "HUMI", "TEMP", "DEWP", "Iws", "PRES"]]
    .dropna()
    .rename(
        columns={
            "HUMI": "humidity",
            "TEMP": "temperature",
            "DEWP": "dew_point",
            "Iws": "instant_wind_speed",
            "PRES": "pressure",
        }
    )
    .copy()
)

feature_df.head(), feature_df.shape

(     precipitation  humidity  temperature  dew_point  instant_wind_speed  \
 12             0.0      32.0         -5.0      -19.0               37.56   
 36             0.0      79.0         -5.0       -8.0               23.69   
 60             1.1      85.0         -9.0      -11.0              105.93   
 84             0.0      43.0        -11.0      -21.0              117.55   
 108            0.0      33.0        -12.0      -25.0               39.35   
 
      pressure  
 12     1015.0  
 36     1026.0  
 60     1021.0  
 84     1030.0  
 108    1034.0  ,
 (2154, 6))

In [3]:
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaled_features = scaler.fit_transform(feature_df)

k_values = range(2, 11)
inertias = []
silhouette_scores = []

for k in k_values:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(scaled_features)
    inertias.append(model.inertia_)
    silhouette_scores.append(silhouette_score(scaled_features, labels))

model_selection_df = pd.DataFrame(
    {
        "k": list(k_values),
        "inertia": inertias,
        "silhouette_score": silhouette_scores,
    }
)

best_k = model_selection_df.loc[model_selection_df["silhouette_score"].idxmax(), "k"]
model_selection_df, best_k

(    k      inertia  silhouette_score
 0   2  8084.541296          0.405922
 1   3  6618.544165          0.420696
 2   4  5303.956390          0.426101
 3   5  4345.049420          0.318673
 4   6  3598.161338          0.335488
 5   7  3160.272303          0.330589
 6   8  2774.783568          0.296625
 7   9  2584.761617          0.291216
 8  10  2372.399192          0.276441,
 np.int64(4))

In [7]:
axes[0].plot(model_selection_df["k"], model_selection_df["inertia"], marker="o", color="tab:blue")
axes[0].set_title("Elbow method")
axes[0].set_xlabel("Number of clusters (k)")
axes[0].set_ylabel("Inertia")
axes[0].grid(True, alpha=0.3)

axes[1].plot(model_selection_df["k"], model_selection_df["silhouette_score"], marker="o", color="tab:green")
axes[1].set_title("Silhouette score")
axes[1].set_xlabel("Number of clusters (k)")
axes[1].set_ylabel("Silhouette score")
axes[1].grid(True, alpha=0.3)

for ax in axes:
    ax.axvline(best_k, color="tab:red", linestyle="--", alpha=0.7)
    ax.text(best_k + 0.05, ax.get_ylim()[1] * 0.92, f"best k = {best_k}", color="tab:red")

plt.tight_layout()
plt.show()

<Figure size 640x480 with 0 Axes>

## Quick interpretation

The elbow plot shows how quickly the within-cluster variance drops as more clusters are added, while the silhouette score shows how well separated the clusters are.

In this analysis, the best choice is the $k$ value with the highest silhouette score. That gives a data-driven starting point for interpreting weather regimes in the Beijing PM dataset.

In [13]:
# K-means analysis with k=4
kmeans_k4 = KMeans(n_clusters=4, random_state=42, n_init=10)
labels_k4 = kmeans_k4.fit_predict(scaled_features)
centroids_k4 = scaler.inverse_transform(kmeans_k4.cluster_centers_)

centroid_df = pd.DataFrame(centroids_k4, columns=feature_df.columns)
centroid_df.round(2)

from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns

pca = PCA(n_components=2)
pcs = pca.fit_transform(feature_df[["precipitation", "humidity", "temperature", "dew_point", "instant_wind_speed", "pressure"]])
feature_df['PC1'], feature_df['PC2'] = pcs[:, 0], pcs[:, 1]

sns.scatterplot(data=feature_df, x='PC1', y='PC2', hue='cluster', 
                palette='Set2', alpha=0.5, s=20)
# Add centroids
centroids = feature_df.groupby('cluster')[['PC1', 'PC2']].mean()
plt.scatter(centroids['PC1'], centroids['PC2'], 
            c=centroids.index, s=200, marker='X', edgecolor='black')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')

ValueError: Shape of passed values is (4, 6), indices imply (4, 8)